In [ ]:
from pathlib import Path
import json
import subprocess
import torch

from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

DEFAULT_MODEL = 'Qwen/Qwen2.5-0.5B-Instruct'

PROJECT_ROOT = Path.cwd()
ADAPTER_PATH = './sft'
MERGED_MODEL_DIR = str(Path(ADAPTER_PATH).parent / "selected_adapter_merged_full")

RUNS = [
    {'task': 'gsm_plus', 'num_fewshot': 0, 'tag': 'zero_shot'},
    {'task': 'gsm_plus', 'num_fewshot': 5, 'tag': 'few_shot_5'},
]

DEVICE = 'cuda:0'
BATCH_SIZE = '8'
SEED = '0,1234,1234,1234'
LIMIT = None  
OUTPUT_ROOT = Path('lm_eval_results_per_task')
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

print('PROJECT_ROOT:', PROJECT_ROOT)
print('DEFAULT_MODEL:', DEFAULT_MODEL)
print('ADAPTER_PATH:', ADAPTER_PATH)
print('MERGED_MODEL_DIR:', MERGED_MODEL_DIR)
print('RUNS:', RUNS)


PROJECT_ROOT: /workspace
DEFAULT_MODEL: Qwen/Qwen2.5-0.5B-Instruct
ADAPTER_PATH: ./sft
MERGED_MODEL_DIR: selected_adapter_merged_full
RUNS: [{'task': 'gsm_plus', 'num_fewshot': 0, 'tag': 'zero_shot'}, {'task': 'gsm_plus', 'num_fewshot': 5, 'tag': 'few_shot_5'}]


In [ ]:
def ensure_merged_model(base_model: str, adapter_path: str, merged_dir: str) -> str:
    adapter = Path(adapter_path)
    merged = Path(merged_dir)

    print('Merging adapter with base model...')
    dtype = torch.float16 if torch.cuda.is_available() else torch.float32
    device_map = 'auto' if torch.cuda.is_available() else None

    base = AutoModelForCausalLM.from_pretrained(
        base_model,
        torch_dtype=dtype,
        device_map=device_map,
        low_cpu_mem_usage=True,
        trust_remote_code=True,
    )
    peft_model = PeftModel.from_pretrained(base, adapter_path)
    merged_model = peft_model.merge_and_unload()

    tokenizer = AutoTokenizer.from_pretrained(base_model, trust_remote_code=True)
    merged.mkdir(parents=True, exist_ok=True)
    merged_model.save_pretrained(merged, safe_serialization=True)
    tokenizer.save_pretrained(merged)

    print(f'Saved merged model: {merged}')
    return str(merged)


MERGED_MODEL_DIR = ensure_merged_model(DEFAULT_MODEL, ADAPTER_PATH, MERGED_MODEL_DIR)


Merging adapter with base model...


config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/988M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

...

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved merged model: selected_adapter_merged_full


In [4]:
def run_lm_eval(model_ref: str, model_tag: str, task: str, num_fewshot: int, run_tag: str) -> Path:
    output_dir = OUTPUT_ROOT / task / run_tag / model_tag
    output_dir.mkdir(parents=True, exist_ok=True)

    model_args = f'pretrained={model_ref},trust_remote_code=True'
    cmd = [
        'lm-eval', 'run',
        '--model', 'hf',
        '--model_args', model_args,
        '--tasks', task,
        '--num_fewshot', str(num_fewshot),
        '--device', DEVICE,
        '--batch_size', BATCH_SIZE,
        '--apply_chat_template',
        '--seed', SEED,
        '--output_path', str(output_dir),
        '--log_samples',
    ]

    if LIMIT is not None:
        cmd.extend(['--limit', str(LIMIT)])

    print('\n$', ' '.join(cmd))
    subprocess.run(cmd, check=True)
    return output_dir


all_outputs = []
for run in RUNS:
    task = run['task']
    shot = int(run['num_fewshot'])
    run_tag = run['tag']

    base_out = run_lm_eval(DEFAULT_MODEL, 'base_qwen2p5_0p5b', task, shot, run_tag)
    merged_out = run_lm_eval(MERGED_MODEL_DIR, 'sft_from_dpo2_merged', task, shot, run_tag)
    all_outputs.append((task, shot, base_out, merged_out))

print('\nDone. Output dirs:')
for item in all_outputs:
    print(item)




$ lm-eval run --model hf --model_args pretrained=Qwen/Qwen2.5-0.5B-Instruct,trust_remote_code=True --tasks gsm_plus --num_fewshot 0 --device cuda:0 --batch_size 8 --apply_chat_template --seed 0,1234,1234,1234 --output_path lm_eval_results_per_task/gsm_plus/zero_shot/base_qwen2p5_0p5b --log_samples


2026-02-22:07:01:59 INFO     [config.evaluate_config:301] Using default fewshot_as_multiturn=True.
2026-02-22:07:02:03 INFO     [_cli.run:376] Selected Tasks: ['gsm_plus']
2026-02-22:07:02:03 INFO     [evaluator:211] Setting random seed to 0 | Setting numpy seed to 1234 | Setting torch manual seed to 1234 | Setting fewshot manual seed to 1234
2026-02-22:07:02:03 INFO     [evaluator:236] Initializing hf model, with arguments: {'pretrained': 'Qwen/Qwen2.5-0.5B-Instruct', 'trust_remote_code': True}
2026-02-22:07:02:04 INFO     [models.huggingface:161] Using device 'cuda:0'
2026-02-22:07:02:05 INFO     [models.huggingface:423] Model parallel was set to False, max memory was not set, and device map was set to {'': 'cuda:0'}
Generating testmini split: 100%|██████████| 2400/2400 [00:00<00:00, 540677.28 examples/s]
2026-02-22:07:02:11 INFO     [tasks:700] Selected tasks:
2026-02-22:07:02:11 INFO     [tasks:691] Task: gsm_plus (gsm_plus/gsm_plus.yaml)
2026-02-22:07:02:11 INFO     [evaluator:314

hf ({'pretrained': 'Qwen/Qwen2.5-0.5B-Instruct'}), gen_kwargs: ({}), limit: None, num_fewshot: 0, batch_size: 8
| Tasks  |Version|     Filter     |n-shot|  Metric   |   |Value |   |Stderr|
|--------|------:|----------------|-----:|-----------|---|-----:|---|-----:|
|gsm_plus|      1|flexible-extract|     0|exact_match|↑  |0.1568|±  |0.0035|
|        |       |strict-match    |     0|exact_match|↑  |0.0000|±  |0.0000|


$ lm-eval run --model hf --model_args pretrained=selected_adapter_merged_full,trust_remote_code=True --tasks gsm_plus --num_fewshot 0 --device cuda:0 --batch_size 8 --apply_chat_template --seed 0,1234,1234,1234 --output_path lm_eval_results_per_task/gsm_plus/zero_shot/sft_from_dpo2_merged --log_samples


2026-02-22:08:28:52 INFO     [config.evaluate_config:301] Using default fewshot_as_multiturn=True.
2026-02-22:08:28:56 INFO     [_cli.run:376] Selected Tasks: ['gsm_plus']
2026-02-22:08:28:56 INFO     [evaluator:211] Setting random seed to 0 | Setting numpy seed to 1234 | Setting torch manual seed to 1234 | Setting fewshot manual seed to 1234
2026-02-22:08:28:56 INFO     [evaluator:236] Initializing hf model, with arguments: {'pretrained': 'selected_adapter_merged_full', 'trust_remote_code': True}
2026-02-22:08:28:57 INFO     [models.huggingface:161] Using device 'cuda:0'
2026-02-22:08:28:58 INFO     [models.huggingface:423] Model parallel was set to False, max memory was not set, and device map was set to {'': 'cuda:0'}
Loading weights: 100%|██████████| 290/290 [00:00<00:00, 1920.38it/s, Materializing param=model.norm.weight]                              
2026-02-22:08:29:01 INFO     [tasks:700] Selected tasks:
2026-02-22:08:29:01 INFO     [tasks:691] Task: gsm_plus (gsm_plus/gsm_plus

hf ({'pretrained': 'selected_adapter_merged_full'}), gen_kwargs: ({}), limit: None, num_fewshot: 0, batch_size: 8
| Tasks  |Version|     Filter     |n-shot|  Metric   |   |Value |   |Stderr|
|--------|------:|----------------|-----:|-----------|---|-----:|---|-----:|
|gsm_plus|      1|flexible-extract|     0|exact_match|↑  |0.1875|±  |0.0038|
|        |       |strict-match    |     0|exact_match|↑  |0.0000|±  |0.0000|


$ lm-eval run --model hf --model_args pretrained=Qwen/Qwen2.5-0.5B-Instruct,trust_remote_code=True --tasks gsm_plus --num_fewshot 5 --device cuda:0 --batch_size 8 --apply_chat_template --seed 0,1234,1234,1234 --output_path lm_eval_results_per_task/gsm_plus/few_shot_5/base_qwen2p5_0p5b --log_samples


2026-02-22:09:40:24 INFO     [config.evaluate_config:301] Using default fewshot_as_multiturn=True.
2026-02-22:09:40:28 INFO     [_cli.run:376] Selected Tasks: ['gsm_plus']
2026-02-22:09:40:28 INFO     [evaluator:211] Setting random seed to 0 | Setting numpy seed to 1234 | Setting torch manual seed to 1234 | Setting fewshot manual seed to 1234
2026-02-22:09:40:28 INFO     [evaluator:236] Initializing hf model, with arguments: {'pretrained': 'Qwen/Qwen2.5-0.5B-Instruct', 'trust_remote_code': True}
2026-02-22:09:40:29 INFO     [models.huggingface:161] Using device 'cuda:0'
2026-02-22:09:40:31 INFO     [models.huggingface:423] Model parallel was set to False, max memory was not set, and device map was set to {'': 'cuda:0'}
Loading weights: 100%|██████████| 290/290 [00:00<00:00, 1610.88it/s, Materializing param=model.norm.weight]                              
2026-02-22:09:40:34 INFO     [tasks:700] Selected tasks:
2026-02-22:09:40:34 INFO     [tasks:691] Task: gsm_plus (gsm_plus/gsm_plus.y

hf ({'pretrained': 'Qwen/Qwen2.5-0.5B-Instruct'}), gen_kwargs: ({}), limit: None, num_fewshot: 5, batch_size: 8
| Tasks  |Version|     Filter     |n-shot|  Metric   |   |Value |   |Stderr|
|--------|------:|----------------|-----:|-----------|---|-----:|---|-----:|
|gsm_plus|      1|flexible-extract|     5|exact_match|↑  |0.2093|±  |0.0040|
|        |       |strict-match    |     5|exact_match|↑  |0.0516|±  |0.0022|


$ lm-eval run --model hf --model_args pretrained=selected_adapter_merged_full,trust_remote_code=True --tasks gsm_plus --num_fewshot 5 --device cuda:0 --batch_size 8 --apply_chat_template --seed 0,1234,1234,1234 --output_path lm_eval_results_per_task/gsm_plus/few_shot_5/sft_from_dpo2_merged --log_samples


2026-02-22:11:09:16 INFO     [config.evaluate_config:301] Using default fewshot_as_multiturn=True.
2026-02-22:11:09:19 INFO     [_cli.run:376] Selected Tasks: ['gsm_plus']
2026-02-22:11:09:19 INFO     [evaluator:211] Setting random seed to 0 | Setting numpy seed to 1234 | Setting torch manual seed to 1234 | Setting fewshot manual seed to 1234
2026-02-22:11:09:19 INFO     [evaluator:236] Initializing hf model, with arguments: {'pretrained': 'selected_adapter_merged_full', 'trust_remote_code': True}
2026-02-22:11:09:20 INFO     [models.huggingface:161] Using device 'cuda:0'
2026-02-22:11:09:21 INFO     [models.huggingface:423] Model parallel was set to False, max memory was not set, and device map was set to {'': 'cuda:0'}
Loading weights: 100%|██████████| 290/290 [00:00<00:00, 1796.20it/s, Materializing param=model.norm.weight]                              
2026-02-22:11:09:24 INFO     [tasks:700] Selected tasks:
2026-02-22:11:09:24 INFO     [tasks:691] Task: gsm_plus (gsm_plus/gsm_plus

hf ({'pretrained': 'selected_adapter_merged_full'}), gen_kwargs: ({}), limit: None, num_fewshot: 5, batch_size: 8
| Tasks  |Version|     Filter     |n-shot|  Metric   |   |Value |   |Stderr|
|--------|------:|----------------|-----:|-----------|---|-----:|---|-----:|
|gsm_plus|      1|flexible-extract|     5|exact_match|↑  |0.2249|±  |0.0041|
|        |       |strict-match    |     5|exact_match|↑  |0.0281|±  |0.0016|


Done. Output dirs:
('gsm_plus', 0, PosixPath('lm_eval_results_per_task/gsm_plus/zero_shot/base_qwen2p5_0p5b'), PosixPath('lm_eval_results_per_task/gsm_plus/zero_shot/sft_from_dpo2_merged'))
('gsm_plus', 5, PosixPath('lm_eval_results_per_task/gsm_plus/few_shot_5/base_qwen2p5_0p5b'), PosixPath('lm_eval_results_per_task/gsm_plus/few_shot_5/sft_from_dpo2_merged'))
